In [23]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Esempio di codice Python per analizzare il corpus BERP (Berkeley Restaurant Project)
presente al link: https://github.com/wooters/berp-trans

Questo script esegue:
1. Lettura del file 'transcript.txt'.
2. Tokenizzazione delle frasi.
3. Calcolo dei conteggi di unigrammi e bigrammi.
4. Calcolo delle probabilità dei bigrammi (modello di catena di Markov).
5. Esempio di calcolo della probabilità di frasi specifiche.

Richiede Python 3.x.
"""

import requests
from collections import defaultdict
import re

# URL dei file raw su GitHub
TRANSCRIPT_URL = "https://raw.githubusercontent.com/wooters/berp-trans/master/transcript.txt"
WORDHIST_URL   = "https://raw.githubusercontent.com/wooters/berp-trans/master/wordhist.txt"

def download_file(url):
    """
    Scarica il contenuto di un file di testo da GitHub (o altra URL) e lo restituisce come lista di righe.
    """
    response = requests.get(url)
    response.raise_for_status()
    text = response.text
    lines = text.strip().split("\n")
    return lines

def tokenize(line):
    """
    Tokenizza la riga esattamente come il comando bash:
      - Divide la riga sul carattere spazio (' ')
      - Rimuove il primo campo
      - Non rimuove token vuoti (rispettando il comportamento di tr)
    """
    parts = line.rstrip("\n").split(" ")
    return parts[1:] if len(parts) > 1 else []

def build_unigram_bigram_counts(transcript_lines):
    """
    Data una lista di righe del transcript, costruisce:
      - un dizionario di conteggio degli unigrammi
      - un dizionario di conteggio dei bigrammi
    La tokenizzazione è eseguita in modo da essere equivalente al comando bash.
    """
    unigram_counts = defaultdict(int)
    bigram_counts = defaultdict(int)
    
    for line in transcript_lines:
        # Utilizza la tokenizzazione "bash-like"
        tokens = tokenize(line)
        
        for i, token in enumerate(tokens):
            unigram_counts[token] += 1
            if i < len(tokens) - 1:
                bigram_counts[(token, tokens[i + 1])] += 1
    
    return unigram_counts, bigram_counts

def compute_bigram_probabilities(unigram_counts, bigram_counts):
    """
    Calcola le probabilità P(w2|w1) = count(w1,w2)/count(w1)
    Restituisce un dizionario: bigram_probs[(w1, w2)] = P(w2|w1).
    """
    bigram_probs = {}
    for (w1, w2), count in bigram_counts.items():
        bigram_probs[(w1, w2)] = count / float(unigram_counts[w1])
    return bigram_probs

def phrase_probability(phrase_tokens, unigram_counts, bigram_probs, total_unigrams):
    """
    Calcola la probabilità di una frase (in forma di lista di token)
    usando il modello di catena di Markov (bigrammi).
    
    Formula semplificata:
      P(t1, t2, ..., tn) ≈ P(t1) * P(t2|t1) * ... * P(tn|t(n-1))
      
    dove P(t1) = count(t1)/total_unigrams
          P(ti|t(i-1)) = bigram_probs.get((t(i-1), ti), 0)
    
    Restituisce un valore float.
    """
    if not phrase_tokens:
        return 0.0
    
    # Probabilità iniziale: P(t1)
    p = unigram_counts[phrase_tokens[0]] / float(total_unigrams) if phrase_tokens[0] in unigram_counts else 0.0
    
    # Moltiplichiamo per ogni bigramma successivo
    for i in range(len(phrase_tokens) - 1):
        w1, w2 = phrase_tokens[i], phrase_tokens[i+1]
        p_next = bigram_probs.get((w1, w2), 0.0)
        p *= p_next
    
    return p

print("Download dei file transcript e wordhist da GitHub...")
transcript_lines = download_file(TRANSCRIPT_URL)
wordhist_lines   = download_file(WORDHIST_URL)

# Salvo i transcript in un nuovo file .txt
print("Salvo i transcript in un nuovo file .txt...")

with open("transcript.txt", 'w') as f:
    f.write("\n".join(transcript_lines))
    f.close()

print("Costruzione dei conteggi di unigrammi e bigrammi dal transcript...")
unigram_counts, bigram_counts = build_unigram_bigram_counts(transcript_lines)

# Calcoliamo le probabilità di bigramma
bigram_probs = compute_bigram_probabilities(unigram_counts, bigram_counts)

# Calcoliamo il totale degli unigrammi
total_unigrams = sum(unigram_counts.values())

# Esempi di frasi da valutare
phrase1 = ["i", "want", "english", "food"]
phrase2 = ["i", "want", "chinese", "food"]

p_phrase1 = phrase_probability(phrase1, unigram_counts, bigram_probs, total_unigrams)
p_phrase2 = phrase_probability(phrase2, unigram_counts, bigram_probs, total_unigrams)

print(f"Probabilità frase '{' '.join(phrase1)}': {p_phrase1:.8f}")
print(f"Probabilità frase '{' '.join(phrase2)}': {p_phrase2:.8f}")

# Esempio: Stampa top 5 unigrammi e top 5 bigrammi più frequenti
print("\nTop 5 Unigrammi (per conteggio):")
top5_uni = sorted(unigram_counts.items(), key=lambda x: x[1], reverse=True)[:5]
for w, c in top5_uni:
    print(f"{w}: {c}")

print("\nTop 5 Bigrammi (per conteggio):")
top5_bi = sorted(bigram_counts.items(), key=lambda x: x[1], reverse=True)[:5]
for (w1, w2), c in top5_bi:
    print(f"({w1}, {w2}): {c}")

Download dei file transcript e wordhist da GitHub...
Salvo i transcript in un nuovo file .txt...
Costruzione dei conteggi di unigrammi e bigrammi dal transcript...
Probabilità frase 'i want english food': 0.00000000
Probabilità frase 'i want chinese food': 0.00005559

Top 5 Unigrammi (per conteggio):
i: 2816
to: 2711
like: 1522
food: 1242
about: 1154

Top 5 Bigrammi (per conteggio):
(like, to): 1172
(i, want): 908
(to, eat): 753
(i, would): 737
(would, like): 711


In [32]:
import requests
from collections import defaultdict

# URL dei file raw su GitHub
TRANSCRIPT_URL = "https://raw.githubusercontent.com/wooters/berp-trans/master/transcript.txt"
WORDHIST_URL   = "https://raw.githubusercontent.com/wooters/berp-trans/master/wordhist.txt"

def download_file(url):
    """
    Scarica il contenuto di un file di testo da GitHub (o altra URL)
    e lo restituisce come lista di righe.
    """
    response = requests.get(url)
    response.raise_for_status()
    text = response.text
    lines = text.strip().split("\n")
    return lines

def tokenize(line):
    """
    Tokenizza la riga esattamente come il comando bash:
      - Divide la riga sul carattere spazio (' ')
      - Rimuove il primo campo
      - Non rimuove token vuoti (rispettando il comportamento di tr)
    """
    parts = line.rstrip("\n").split(" ")
    return parts[1:] if len(parts) > 1 else []

def build_unigram_bigram_counts(transcript_lines):
    """
    Data una lista di righe del transcript, costruisce:
      - un dizionario di conteggio degli unigrammi
      - un dizionario di conteggio dei bigrammi
    La tokenizzazione viene eseguita in modo "bash-like" e
    viene aggiunto <s> all'inizio e </s> alla fine di ogni frase.
    """
    unigram_counts = defaultdict(int)
    bigram_counts = defaultdict(int)
    
    for line in transcript_lines:
        # Tokenizza la riga e aggiunge i boundary
        tokens = tokenize(line)
        tokens = ["<s>"] + tokens + ["</s>"]
        
        for i, token in enumerate(tokens):
            unigram_counts[token] += 1
            if i < len(tokens) - 1:
                bigram_counts[(token, tokens[i + 1])] += 1
    
    return unigram_counts, bigram_counts

def compute_bigram_probabilities(unigram_counts, bigram_counts):
    """
    Calcola le probabilità P(w2|w1) = count(w1,w2)/count(w1)
    Restituisce un dizionario: bigram_probs[(w1, w2)] = P(w2|w1).
    """
    bigram_probs = {}
    for (w1, w2), count in bigram_counts.items():
        bigram_probs[(w1, w2)] = count / float(unigram_counts[w1])
    return bigram_probs

def phrase_probability(phrase_tokens, bigram_probs):
    """
    Calcola la probabilità di una frase (data come lista di token senza boundary)
    usando il modello di catena di Markov (bigrammi). Aggiunge i token
    di inizio (<s>) e fine frase (</s>) e calcola:
      P(phrase) = P(t1|<s>) * P(t2|t1) * ... * P(</s>|t_n)
      
    Restituisce un valore float.
    """
    # Aggiunge i token di inizio e fine frase
    tokens = ["<s>"] + phrase_tokens + ["</s>"]
    prob = 1.0
    for i in range(len(tokens) - 1):
        bigram = (tokens[i], tokens[i+1])
        p_bigram = bigram_probs.get(bigram, 0.0)
        prob *= p_bigram
    return prob

# Download dei file transcript e wordhist da GitHub
print("Download dei file transcript e wordhist da GitHub...")
transcript_lines = download_file(TRANSCRIPT_URL)
wordhist_lines   = download_file(WORDHIST_URL)

# Salvo i transcript in un nuovo file .txt
print("Salvo i transcript in un nuovo file .txt...")
with open("transcript.txt", 'w', encoding="utf-8") as f:
    f.write("\n".join(transcript_lines))

print("Costruzione dei conteggi di unigrammi e bigrammi dal transcript...")
unigram_counts, bigram_counts = build_unigram_bigram_counts(transcript_lines)

# Calcoliamo le probabilità di bigramma
bigram_probs = compute_bigram_probabilities(unigram_counts, bigram_counts)

# Esempi di frasi da valutare (senza boundary, che verranno aggiunti nella funzione)
phrase1 = ["i", "want", "english", "food"]
phrase2 = ["i", "want", "chinese", "food"]

p_phrase1 = phrase_probability(phrase1, bigram_probs)
p_phrase2 = phrase_probability(phrase2, bigram_probs)

print(f"Probabilità frase '{' '.join(phrase1)}': {p_phrase1:.8f}")
print(f"Probabilità frase '{' '.join(phrase2)}': {p_phrase2:.8f}")

# Esempio: Stampa top 5 unigrammi e top 5 bigrammi più frequenti
print("\nTop 7 Unigrammi (per conteggio):")
top5_uni = sorted(unigram_counts.items(), key=lambda x: x[1], reverse=True)[:7]
for w, c in top5_uni:
    print(f"{w}: {c}")

print("\nTop 5 Bigrammi (per conteggio):")
top5_bi = sorted(bigram_counts.items(), key=lambda x: x[1], reverse=True)[:5]
for (w1, w2), c in top5_bi:
    print(f"({w1}, {w2}): {c}")

Download dei file transcript e wordhist da GitHub...
Salvo i transcript in un nuovo file .txt...
Costruzione dei conteggi di unigrammi e bigrammi dal transcript...
Probabilità frase 'i want english food': 0.00000000
Probabilità frase 'i want chinese food': 0.00016241

Top 7 Unigrammi (per conteggio):
<s>: 8566
</s>: 8566
i: 2816
to: 2711
like: 1522
food: 1242
about: 1154

Top 5 Bigrammi (per conteggio):
(<s>, i): 1922
(like, to): 1172
(i, want): 908
(food, </s>): 806
(to, eat): 753


In [39]:
import pandas as pd

def generate_bigram_table(bigram_counts, words):
    """
    Genera una tabella (DataFrame) contenente i conteggi dei bigrammi per le parole specificate,
    includendo i token di inizio frase <s> e di fine frase </s>.
    
    Parametri:
      - bigram_counts: dizionario dei bigrammi (chiave = (w1, w2), valore = conteggio)
      - words: lista di parole (stringhe) da includere come righe e colonne
      
    Restituisce:
      - df: DataFrame con righe e colonne corrispondenti alle parole (comprese <s> e </s>),
            contenente i conteggi dei bigrammi.
    """
    # Assicuriamoci di includere i token di inizio e fine frase
    if "<s>" not in words:
        words = ["<s>"] + words
    if "</s>" not in words:
        words = words + ["</s>"]
        
    table = {}
    for w1 in words:
        row = {}
        for w2 in words:
            row[w2] = bigram_counts.get((w1, w2), 0)
        table[w1] = row
    df = pd.DataFrame.from_dict(table, orient='index')
    # Riordina le righe e le colonne in base all'ordine della lista "words"
    df = df.loc[words, words]
    return df

# Definisci l'insieme di parole da visualizzare nella tabella
words = ["i", "want", "to", "eat", "chinese", "food", "lunch", "spend"]

# Genera la tabella dei bigrammi utilizzando i conteggi ottenuti precedentemente (bigram_counts)
df_bigram = generate_bigram_table(bigram_counts, words)

# Stampa la tabella in formato Markdown
df_bigram.head(10)

,<s>,i,want,to,eat,chinese,food,lunch,spend,</s>
<s>,0,1922,4,32,4,10,4,39,1,0
i,0,1,908,0,12,0,0,0,2,0
want,0,2,0,673,0,7,6,6,1,2
to,0,0,0,2,753,3,0,6,233,3
eat,0,0,0,0,0,16,2,52,0,10
chinese,0,4,0,0,0,0,99,1,0,10
food,0,14,0,13,0,0,0,0,0,806
lunch,0,1,0,0,0,0,1,0,0,221
spend,0,0,0,1,0,0,0,0,0,8
</s>,0,0,0,0,0,0,0,0,0,0


In [43]:
for x in ["<s>", "i", "want", "to", "eat", "chinese", "food", "lunch", "spend", "</s>"]:
    print(x, unigram_counts[x])

<s> 8566
i 2816
want 1038
to 2711
eat 829
chinese 193
food 1242
lunch 392
spend 310
</s> 8566


In [40]:
# Applicazione del Laplace Smoothing

def apply_laplace_smoothing(df):
    """
    Applica il Laplace smoothing (add-one smoothing) al DataFrame dei conteggi dei bigrammi.
    Aggiunge 1 ad ogni conteggio e normalizza ogni riga per ottenere delle probabilità condizionali.
    
    Parametri:
      - df: DataFrame con conteggi dei bigrammi (righe: parola corrente, colonne: parola successiva)
      
    Restituisce:
      - df_smoothed: DataFrame con le probabilità condizionali smoothed.
    """
    # Aggiunge 1 a ciascun conteggio
    df_smoothed = df + 1
    # Normalizza ogni riga (somma riga = 1)
    df_smoothed = df_smoothed.div(df_smoothed.sum(axis=1), axis=0)
    return df_smoothed

# Definisci l'insieme di parole da visualizzare nella tabella
words = ["i", "want", "to", "eat", "chinese", "food", "lunch", "spend"]

# Genera la tabella dei bigrammi utilizzando i conteggi ottenuti precedentemente (bigram_counts)
df_bigram = generate_bigram_table(bigram_counts, words)

# Applica il Laplace smoothing
df_bigram_smoothed = apply_laplace_smoothing(df_bigram)

# Stampa la tabella in formato Markdown (le prime 8 righe)
df_bigram_smoothed.head(10)

,<s>,i,want,to,eat,chinese,food,lunch,spend,</s>
<s>,0.000494,0.949161,0.002468,0.016288,0.002468,0.005429,0.002468,0.019743,0.000987,0.000494
i,0.001072,0.002144,0.974277,0.001072,0.013934,0.001072,0.001072,0.001072,0.003215,0.001072
want,0.001414,0.004243,0.001414,0.953324,0.001414,0.011315,0.009901,0.009901,0.002829,0.004243
to,0.000990,0.000990,0.000990,0.002970,0.746535,0.003960,0.000990,0.006931,0.231683,0.003960
eat,0.011111,0.011111,0.011111,0.011111,0.011111,0.188889,0.033333,0.588889,0.011111,0.122222
chinese,0.008065,0.040323,0.008065,0.008065,0.008065,0.008065,0.806452,0.016129,0.008065,0.088710
food,0.001186,0.017794,0.001186,0.016607,0.001186,0.001186,0.001186,0.001186,0.001186,0.957295
lunch,0.004292,0.008584,0.004292,0.004292,0.004292,0.004292,0.008584,0.004292,0.004292,0.952790
spend,0.052632,0.052632,0.052632,0.105263,0.052632,0.052632,0.052632,0.052632,0.052632,0.473684
</s>,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000


In [41]:
def sentence_probability(sentence, smoothed_df):
    """
    Calcola la probabilità di una frase come prodotto delle probabilità condizionali dei bigrammi.
    
    Parametri:
      - sentence: stringa contenente la frase.
      - smoothed_df: DataFrame contenente le probabilità condizionali (smoothed) dei bigrammi.
      
    Restituisce:
      - prob: probabilità della frase.
    """
    tokens = sentence.split()  # Tokenizzazione semplice basata sugli spazi
    prob = 1.0
    # Calcola il prodotto delle probabilità per ogni bigramma nella frase
    for i in range(len(tokens) - 1):
        w1 = tokens[i]
        w2 = tokens[i + 1]
        if w1 in smoothed_df.index and w2 in smoothed_df.columns:
            prob *= smoothed_df.loc[w1, w2]
        else:
            # Se una parola non è presente nel vocabolario, la probabilità è 0
            return 0.0
    return prob

# Esempio di frasi
sentence1 = phrase1 = " ".join(["i", "want", "english", "food"]) #"i want to eat chinese food"
sentence2 = phrase2 = " ".join(["i", "want", "chinese", "food"]) #"i want to spend lunch"

# Calcola le probabilità delle frasi usando il dataframe smoothed dei bigrammi
prob_sentence1 = sentence_probability(sentence1, df_bigram_smoothed)
prob_sentence2 = sentence_probability(sentence2, df_bigram_smoothed)

print("Probability of sentence1:", prob_sentence1)
print("Probability of sentence2:", prob_sentence2)

Probability of sentence1: 0.0
Probability of sentence2: 0.008890601152814617
